# 60 — Weak Bullet Rewriter
**Goal:** Selectively rewrite only low-scoring resume bullets using LLM.

Ch. 37 built a STAR scorer; this chapter adds the LLM as a **surgical editor**: score every bullet, send only the weak ones to the model, keep the strong ones verbatim, and let a human accept or reject each rewrite. The LLM is not rewriting the resume — it is proposing fixes for the bullets that need them.

**Why it matters for resumes / ATS:** rewriting everything costs money and, worse, erases the candidate's voice. Selective rewriting spends LLM budget only where the rule-based scorer found measurable weakness, so the output stays authentic where it was already good — and the accept/reject step keeps a human in the loop over text that will represent a real person.

## 1. Detection-First Strategy

The pipeline is deliberately asymmetric: cheap deterministic scoring runs on **every** bullet; expensive LLM rewriting runs on **few**. Score with the Ch. 37 STAR scorer, send only bullets below the threshold (here `< 0.5`), preserve the rest, then review proposed changes.

**What the code does:** prints the five-step pipeline (score — filter — rewrite — preserve — review) and its three motivations:
- **Cost-effective** — the notebook estimates only ~40% of bullets typically need rewriting, so the LLM bill roughly halves.
- **Preserves authentic writing** — strong bullets keep the candidate's original words.
- **Focuses the LLM** — model effort lands where it measurably adds value.

The threshold is the knob: lower it and you save money but ship weaker text; raise it and quality improves at linear cost.

In [ ]:
print('''Weak bullet rewriter pipeline:
1. Score each bullet using the STAR scorer (Notebook 37)
2. Only send weak bullets (score < 0.5) to LLM
3. Preserve strong bullets as-is
4. Review and accept/reject changes

Pros:
- Cost-effective (only rewrite ~40% of bullets)
- Preserves authentic writing
- Focuses LLM effort where it adds value''')

## 2. Bullet Scoring + Selective Rewrite

`score_bullet()` is a transparent 0—1 heuristic: three independent quality signals, each adding a fixed amount, capped at 1.0. No model, no magic — a recruiter can read why a bullet scored low.

**What the code does:**
- `+0.3` if the first word is in `ACTION_VERBS` (`developed`, `led`, `reduced`, ...) — the single strongest signal of a good bullet.
- `+0.3` if the text contains a quantified result — regex for `%`, `million`, `billion`, `$`.
- `+0.15` if a tool is mentioned after `using`/`with`/`via` + a capital letter.

**Expected (verified by running):** all three sample bullets score below the 0.5 threshold (`0.30`, `0.00`, `0.00`), so all are sent to the LLM — including "Reduced model latency by 40% through TensorFlow optimization". The reason is a live bug: the regexes are written `r"\\d+\\s*(%|million|billion|\\$)"` and `r"(using|with|via)\\s+[A-Z]"`, where the doubled backslash in a raw string means a *literal backslash* — so the quantified-result and tool-mention checks never fire, and the strongest bullet scores 0.30. With the intended `\d`/`\s` patterns it would score 0.75 and be skipped. A scoring bug that under-scores is cheap; one that over-scores ships bad text.

In [ ]:
import re

ACTION_VERBS = {"developed", "led", "reduced", "built", "designed", "implemented",
                "created", "managed", "delivered", "achieved", "improved"}

def score_bullet(bullet):
    """Score a bullet 0-1 based on quality signals."""
    text = bullet.strip()
    score = 0
    first_word = text.split()[0].lower().rstrip(",.;:")
    if first_word in ACTION_VERBS: score += 0.3
    if re.search(r"\\d+\\s*(%|million|billion|\\$)", text, re.IGNORECASE): score += 0.3
    if re.search(r"(using|with|via)\\s+[A-Z]", text): score += 0.15
    return min(score, 1.0)

bullets = [
    "Reduced model latency by 40% through TensorFlow optimization",
    "Was responsible for ML model development",
    "Worked on various data pipeline tasks",
]
for b in bullets:
    score = score_bullet(b)
    action = "SKIP (strong)" if score >= 0.5 else "→ SEND TO LLM"
    print(f"  {score:.2f} {action:16s}: {b[:50]}")

## 3. LLM Rewrite Prompt

The rewrite prompt is the whole product in one template: it states the output contract (STAR), the style constraints (strong verb, quantified result, specific technology), and the input slot. Nothing else — no examples here, because Ch. 56 showed few-shot belongs in the template registry, and this prompt is short enough to inline.

**What the code does:** defines `BULLET_REWRITE_PROMPT` with an `{bullet}` slot, prints "Prompt template ready.", and prints an example rewrite in the source (`Architected and deployed ML models achieving 95% accuracy, reducing manual review time by 40% using TensorFlow`).

**Expected (with a key):** sending "Was responsible for ML model development" through the template returns a rewritten bullet that starts with an action verb and carries a quantified result — the exact transformation Ch. 56's `bullet_rewrite` template demonstrated. Without a key the cell is inert by design: it documents the contract rather than calling the API.

In [ ]:
BULLET_REWRITE_PROMPT = """Rewrite the following resume bullet point to follow STAR format:
- Start with a strong action verb
- Include a quantified result where possible
- Be specific about technology and context

Original: {bullet}
Rewritten:"""

print("Prompt template ready.")
print("With API key, send to model and get rewritten version.")
print("\nExample rewrite:")
print("  Input:  'Was responsible for ML model development'")
print("  Output: 'Architected and deployed ML models achieving 95% accuracy, reducing manual review time by 40% using TensorFlow'")

## Summary: Selective rewriting cuts costs and preserves authentic writing. Only rewrite what's weak.

**Detect cheap, rewrite rarely — the LLM edits only where the scorer found weakness.**

Rule-based scoring filters every bullet; the model rewrites only those under threshold; humans accept or reject each proposal. That asymmetry keeps the LLM bill proportional to the *problem*, not the resume length, and preserves authentic writing in the bullets that were already strong.

The rewrites this chapter proposes are generated in STAR shape — which is exactly the format Ch. 61 industrializes, generating full STAR bullets from raw experience data.